# 3.1 — La rétropropagation : la chaîne des gradients à la main

**Navigation** : [<< 2.2-Descente-de-gradient](../02-ML-Cours/2.2-Descente-de-gradient.ipynb) | [2.9-Grokking (épilogue du socle ML)](../02-ML-Cours/2.9-Grokking-Generalisation.ipynb) | [Index](README.md) | [3.2 — Les optimisateurs >>](3.2-Optimisateurs.ipynb)

**Kernel** : .NET C# (MathNet et TorchSharp) · **CPU** : oui · **Durée** : selon la machine

***

## Introduction

Le socle [`02-ML-Cours`](../02-ML-Cours/README.md) s'arrête sur un paradoxe. D'un côté, `2.2` a
ouvert la boîte noire de `fit()` — mais pour **une droite** : un modèle à deux paramètres, un
gradient qui s'écrit sur une ligne. De l'autre, `2.9` a entraîné un vrai réseau de neurones
(PyTorch) pour montrer le grokking — en le laissant entièrement **boîte noire** :
`loss.backward()` et l'entraînement suit. Entre les deux, le chaînon manquant : **comment le
gradient traverse-t-il les couches d'un réseau ?**

Ce notebook le construit à la main. Un petit perceptron multi-couches (MLP) en **C# avec MathNet.Numerics,
sans autograd**, sur un problème 2D non séparable linéairement. La démarche en trois contrôles :

1. **le contrôle du calcul** : le gradient écrit à la main (chain rule couche par couche) est
   confronté au gradient numérique par différence finie — l'écart doit être nul à l'epsilon près ;
2. **la comparaison avec autograd** : la même architecture, les mêmes poids initiaux, le même pas
   de descente en PyTorch — écarts de perte et de trajectoire mesurés ;
3. **l’expérience d’initialisation nulle** : sur ce réseau ReLU et ce split équilibré, l’initialisation nulle laisse les poids cachés inactifs ; une initialisation asymétrique peut briser la symétrie.

Littérature fondatrice : Rumelhart, Hinton & Williams (1986), *Learning representations by
back-propagating errors* — l'article qui a generalisé la chain rule aux réseaux multi-couches.

**Objectifs** : dériver et vérifier les gradients, mesurer les entraînements et comparer à l’autograd. **Prérequis** : algèbre matricielle et dérivation. **Durée** : 2–3 heures. [Version Python](3.1-Retropropagation.ipynb).


Installer TorchSharp-cpu 0.107.0, MathNet.Numerics et CPython avec scikit-learn ; définir `PYTHONNET_PYDLL` et, si nécessaire, `PYTHONHOME` et `PYTHONPATH`. Python.NET sert aux jeux de données et à la baseline sklearn.


***

## 1. Le problème : une frontière courbe que la régression logistique ne peut pas tracer

Le jeu de données `make_moons` (déjà croisé en [2.7](../02-ML-Cours/2.7-Modeles-Non-Parametriques.ipynb))
compose deux demi-lunes imbriquées : aucune droite ne sépare les deux classes. C'est le terrain
où un modèle **linéaire** se heurte à sa frontière droite. Les réseaux et d’autres modèles non linéaires peuvent représenter des frontières courbes.

In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
#r "nuget: TorchSharp-cpu, 0.107.0"
#r "nuget: PythonNet, 3.0.5"
using System; using System.Linq; using System.Collections.Generic; using System.Text; using System.Globalization; using System.Net;
using MathNet.Numerics.LinearAlgebra; using MathNet.Numerics.LinearAlgebra.Double; using TorchSharp; using static TorchSharp.torch; using Python.Runtime; using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture=CultureInfo.InvariantCulture;torch.set_num_threads(1);
if(!PythonEngine.IsInitialized){var dll=Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");if(string.IsNullOrWhiteSpace(dll))throw new Exception("Définir PYTHONNET_PYDLL vers la bibliothèque partagée CPython");Runtime.PythonDLL=dll;PythonEngine.Initialize();PythonEngine.BeginAllowThreads();}
record PlotSvg(string Svg);
Formatter.Register<PlotSvg>((p,w)=>w.Write(p.Svg),"text/html");
string F(double d)=>d.ToString("0.###",CultureInfo.InvariantCulture);
string Esc(string s)=>WebUtility.HtmlEncode(s);
Matrix<double> FromRows(double[][] rows)=>DenseMatrix.OfRowArrays(rows);
double[][] xtrRows=null,xteRows=null;long[] ytr=null,yte=null;double[] coef=null;double intercept=0,linearTrain=0,linearTest=0;
using(Py.GIL()){
 dynamic ds=Py.Import("sklearn.datasets"),ms=Py.Import("sklearn.model_selection"),lin=Py.Import("sklearn.linear_model"),np=Py.Import("numpy"),sklearn=Py.Import("sklearn"),sys=Py.Import("sys");
 using PyObject pair=ds.make_moons(n_samples:600,noise:.20,random_state:42);using var X=pair.GetItem(0);using var y=pair.GetItem(1);using PyObject split=ms.train_test_split(X,y,test_size:.25,random_state:42,stratify:y);
 using var tr=split.GetItem(0);using var te=split.GetItem(1);using var yt=split.GetItem(2);using var ye=split.GetItem(3);dynamic model=lin.LogisticRegression();model.fit(tr,yt);linearTrain=(double)model.score(tr,yt);linearTest=(double)model.score(te,ye);
 using var a=tr.InvokeMethod("tolist");xtrRows=a.As<double[][]>();using var b=te.InvokeMethod("tolist");xteRows=b.As<double[][]>();using var c=yt.InvokeMethod("tolist");ytr=c.As<long[]>();using var d=ye.InvokeMethod("tolist");yte=d.As<long[]>();using PyObject e=model.coef_.tolist();coef=e.As<double[][]>()[0];using PyObject f=model.intercept_.tolist();intercept=f.As<double[]>()[0];
 Console.WriteLine($".NET {Environment.Version} | CPython {sys.version} | NumPy {np.__version__} | sklearn {sklearn.__version__} | Python.NET {typeof(PythonEngine).Assembly.GetName().Version} | TorchSharp {typeof(Tensor).Assembly.GetName().Version}");
}
var Xtr=FromRows(xtrRows);var Xte=FromRows(xteRows);Console.WriteLine($"make_moons 600, split stratifié {Xtr.RowCount}/{Xte.RowCount} ; LogisticRegression train {linearTrain:F3} test {linearTest:F3}");
// Repères des figures communs aux jeux : limites et ticks.
double xmin=xtrRows.Concat(xteRows).Min(r=>r[0])-.5,xmax=xtrRows.Concat(xteRows).Max(r=>r[0])+.5,ymin=xtrRows.Concat(xteRows).Min(r=>r[1])-.5,ymax=xtrRows.Concat(xteRows).Max(r=>r[1])+.5;
string Begin(int width,int height,string title)=>$"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}' viewBox='0 0 {width} {height}'><rect width='{width}' height='{height}' fill='white'/><text x='{width/2}' y='23' text-anchor='middle' font-family='sans-serif' font-size='15'>{Esc(title)}</text>";
string Axes(double x,double y,double w,double hh,string xlabel,string ylabel,double xmin0,double xmax0,double ymin0,double ymax0){var s=new StringBuilder();s.Append($"<rect x='{F(x)}' y='{F(y)}' width='{F(w)}' height='{F(hh)}' fill='white' stroke='#333'/>");for(int k=0;k<=4;k++){double px=x+w*k/4,py=y+hh-hh*k/4;s.Append($"<text x='{F(px)}' y='{F(y+hh+16)}' text-anchor='middle' font-size='10'>{F(xmin0+(xmax0-xmin0)*k/4)}</text><text x='{F(x-6)}' y='{F(py+3)}' text-anchor='end' font-size='10'>{F(ymin0+(ymax0-ymin0)*k/4)}</text>");}s.Append($"<text x='{F(x+w/2)}' y='{F(y+hh+32)}' text-anchor='middle' font-size='11'>{Esc(xlabel)}</text><text transform='translate({F(x-40)} {F(y+hh/2)}) rotate(-90)' text-anchor='middle' font-size='11'>{Esc(ylabel)}</text>");return s.ToString();}
string Decision(string title,double[][] rows,long[] labels,Func<double,double,double> score){double x=70,y=55,w=500,hh=330;var s=new StringBuilder(Begin(640,450,title));s.Append(Axes(x,y,w,hh,"x1","x2",xmin,xmax,ymin,ymax));double PX(double a)=>x+(a-xmin)/(xmax-xmin)*w;double PY(double b)=>y+hh-(b-ymin)/(ymax-ymin)*hh;for(int i=0;i<rows.Length;i++)s.Append($"<circle cx='{F(PX(rows[i][0]))}' cy='{F(PY(rows[i][1]))}' r='2.5' fill='{(labels[i]==0?"#3168b0":"#d34b49")}' opacity='.7'/>");int n=200;for(int i=0;i<n-1;i++)for(int j=0;j<n-1;j++){double x0=xmin+(xmax-xmin)*i/(n-1),x1=xmin+(xmax-xmin)*(i+1)/(n-1),y0=ymin+(ymax-ymin)*j/(n-1),y1=ymin+(ymax-ymin)*(j+1)/(n-1);double a=score(x0,y0),b=score(x1,y0),c=score(x0,y1);if((a>=0)!=(b>=0))s.Append($"<circle cx='{F(PX((x0+x1)/2))}' cy='{F(PY(y0))}' r='1' fill='#222'/>");if((a>=0)!=(c>=0))s.Append($"<circle cx='{F(PX(x0))}' cy='{F(PY((y0+y1)/2))}' r='1' fill='#222'/>");}s.Append("<text x='75' y='440' font-size='11' fill='#3168b0'>● classe 0</text><text x='160' y='440' font-size='11' fill='#d34b49'>● classe 1</text><text x='250' y='440' font-size='11'>● frontière de décision</text></svg>");return s.ToString();}
string Curve(string title,List<(int,double)> first,List<(int,double)> second=null){int width=640,height=450;double x=70,y=55,w=500,hh=330;var all=second==null?first:first.Concat(second).ToList();double low=Math.Min(0,all.Min(p=>p.Item2)),high=all.Max(p=>p.Item2)*1.05,maxit=Math.Max(1,all.Max(p=>p.Item1));var s=new StringBuilder(Begin(width,height,title));s.Append(Axes(x,y,w,hh,"itération","cross-entropy",0,maxit,low,high));string Path(List<(int,double)> data,string color)=>"<polyline fill='none' stroke='"+color+"' stroke-width='2' points='"+string.Join(" ",data.Select(p=>F(x+w*p.Item1/maxit)+","+F(y+hh-(p.Item2-low)/(high-low)*hh)))+"'/>";s.Append(Path(first,"#3168b0"));if(second!=null)s.Append(Path(second,"#d34b49"));s.Append("<text x='80' y='440' font-size='11' fill='#3168b0'>— manuel</text>");if(second!=null)s.Append("<text x='180' y='440' font-size='11' fill='#d34b49'>— TorchSharp autograd</text>");return s+"</svg>";}
display(new PlotSvg(Decision("Données d’entraînement et frontière logistique",xtrRows,ytr,(x,y)=>coef[0]*x+coef[1]*y+intercept)));

Installed Packages MathNet.Numerics, 5.0.0 PythonNet, 3.0.5 TorchSharp-cpu, 0.107.0

.NET 10.0.14 | CPython 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | NumPy 2.4.4 | sklearn 1.9.1 | Python.NET 3.0.5.0 | TorchSharp 0.107.0.0


make_moons 600, split stratifié 450/150 ; LogisticRegression train 0.871 test 0.873


Données d’entraînement et frontière logistique -1.865 -1.531 -0.668 -0.637 0.53 0.258 1.727 1.153 2.924 2.047 x1 x2 ● classe 0 ● classe 1 ● frontière de décision

**Lecture.** La régression logistique réelle imprimée ci-dessus fournit une frontière linéaire et une précision test mesurée sur 150 points ; une frontière courbe peut capturer une structure différente, sans garantir un résultat universel.


***

## 2. Le MLP en C# avec MathNet.Numerics : forward pass matricielle

Architecture minimale mais complète : **2 entrées → 32 neurones cachés (ReLU) → 2 sorties
(softmax)**. Tout est écrit en algèbre linéaire explicite — chaque couche est une matrice de
poids `W` et un vecteur de biais `b` :

$$Z^{(1)} = XW^{(1)} + b^{(1)}, \quad A^{(1)} = \mathrm{ReLU}(Z^{(1)}), \quad
Z^{(2)} = A^{(1)}W^{(2)} + b^{(2)}, \quad P = \mathrm{softmax}(Z^{(2)})$$

Le coût est l'entropie croisée moyenne : $L = -\frac{1}{n}\sum_i \log P_{i, y_i}$. L'init
**He** ($W \sim \mathcal{N}(0, \sqrt{2/n_{in}})$) est celle d'origine pour ReLU — le §7 montre
ce qui arrive sans elle.

In [2]:
class Params { public Matrix<double> W1,W2; public Vector<double> b1,b2; public Params Copy()=>new Params{W1=W1.Clone(),W2=W2.Clone(),b1=b1.Clone(),b2=b2.Clone()}; }
class ForwardResult { public Matrix<double> Z1,A1,Z2,P; }
Random random=new Random(42);double Gaussian(){double u=1-random.NextDouble(),v=1-random.NextDouble();return Math.Sqrt(-2*Math.Log(u))*Math.Cos(2*Math.PI*v);}
Params He(){return new Params{W1=DenseMatrix.Create(2,32,(i,j)=>Gaussian()*Math.Sqrt(2.0/2)),b1=DenseVector.Create(32,0),W2=DenseMatrix.Create(32,2,(i,j)=>Gaussian()*Math.Sqrt(2.0/32)),b2=DenseVector.Create(2,0)};}
ForwardResult Forward(Params p,Matrix<double> x){var z1=x*p.W1;for(int i=0;i<z1.RowCount;i++)for(int j=0;j<z1.ColumnCount;j++)z1[i,j]+=p.b1[j];var a=z1.Map(v=>Math.Max(0,v));var z2=a*p.W2;for(int i=0;i<z2.RowCount;i++)for(int j=0;j<2;j++)z2[i,j]+=p.b2[j];var probs=DenseMatrix.Create(z2.RowCount,2,0);for(int i=0;i<z2.RowCount;i++){double m=Math.Max(z2[i,0],z2[i,1]);double e0=Math.Exp(z2[i,0]-m),e1=Math.Exp(z2[i,1]-m);probs[i,0]=e0/(e0+e1);probs[i,1]=e1/(e0+e1);}return new ForwardResult{Z1=z1,A1=a,Z2=z2,P=probs};}
double Loss(Params p,Matrix<double> x,long[] labels){var z=Forward(p,x).Z2;double sum=0;for(int i=0;i<x.RowCount;i++){double m=Math.Max(z[i,0],z[i,1]);sum+=Math.Log(Math.Exp(z[i,0]-m)+Math.Exp(z[i,1]-m))-(z[i,(int)labels[i]]-m);}return sum/x.RowCount;}
var initial=He();var out0=Forward(initial,Xtr.SubMatrix(0,5,0,2));Console.WriteLine($"MLP 2→32→2, 162 paramètres ; P {out0.P.RowCount}×{out0.P.ColumnCount}, première somme={out0.P.Row(0).Sum():F6} ; loss initiale={Loss(initial,Xtr,ytr):F6}");
var extreme=initial.Copy();extreme.W1.Clear();extreme.W2.Clear();extreme.b1.Clear();extreme.b2[0]=1000;extreme.b2[1]=-1000;var one=DenseMatrix.Create(1,2,0);if(Loss(extreme,one,new long[]{0})!=0||Loss(extreme,one,new long[]{1})!=2000)throw new Exception("Cross-entropy instable");extreme.b2[0]=1e20;extreme.b2[1]=1e20;double common=Loss(extreme,one,new long[]{0});if(Math.Abs(common-Math.Log(2))>1e-15)throw new Exception("Annulation du décalage commun");extreme.b2[0]=1e308;extreme.b2[1]=-1e308;if(Loss(extreme,one,new long[]{0})!=0)throw new Exception("Classe maximale ±1e308");extreme.b2[0]=1e20;extreme.b2[1]=1e20;Console.WriteLine($"Logits extrêmes : 0, 2000, ln(2) avec décalage ({common:F12}) et classe maximale ±1e308 : 0");

MLP 2→32→2, 162 paramètres ; P 5×2, première somme=1.000000 ; loss initiale=1.023154


Logits extrêmes : 0, 2000, ln(2) avec décalage (0.693147180560) et classe maximale ±1e308 : 0


**Lecture.** La loss initiale et les sommes de probabilités sont affichées par le calcul C# ; la graine C# ne reproduit pas les tirages NumPy. Les logits extrêmes contrôlent la stabilité de la perte.


***

## 3. Backward pass à la main : la chain rule couche par couche

Le principe : la dérivée de $L$ par rapport à un poids profond est un **produit de dérivées
locales** le long du chemin qui relie la sortie à ce poids. Pour la couche de sortie avec
softmax + entropie croisée, les deux se simplifient remarquablement :

$$\frac{\partial L}{\partial Z^{(2)}} = \frac{1}{n}\left(P - Y\right)$$

Puis on remonte : chaque couche cachee propage le gradient d'aval via $W$ et multiplie par la
dérivée locale de sa fonction d'activation ($\mathrm{ReLU}'(z) = \mathbb{1}[z > 0]$) :

$$\frac{\partial L}{\partial W^{(2)}} = A^{(1)\top}\delta^{(2)}, \quad
\delta^{(1)} = \left(\delta^{(2)}W^{(2)\top}\right) \odot \mathrm{ReLU}'(Z^{(1)}), \quad
\frac{\partial L}{\partial W^{(1)}} = X^\top\delta^{(1)}$$

avec $\delta^{(\ell)} = \frac{\partial L}{\partial Z^{(\ell)}}$. C'est le **théorème de
différentiation en chaîne appliqué à un graphe** : coûte un seul forward + un backward, quel que
soit le nombre de couches.

In [3]:
Params Backward(Params p,Matrix<double> x,long[] labels){var f=Forward(p,x);var dz=f.P.Clone();for(int i=0;i<x.RowCount;i++)dz[i,(int)labels[i]]-=1;dz/=x.RowCount;var w2=f.A1.TransposeThisAndMultiply(dz);var b2=DenseVector.Create(2,j=>dz.Column(j).Sum());var dz1=dz*p.W2.Transpose();for(int i=0;i<dz1.RowCount;i++)for(int j=0;j<dz1.ColumnCount;j++)if(f.Z1[i,j]<=0)dz1[i,j]=0;return new Params{W1=x.TransposeThisAndMultiply(dz1),b1=DenseVector.Create(32,j=>dz1.Column(j).Sum()),W2=w2,b2=b2};}
var grad=Backward(initial,Xtr,ytr);Console.WriteLine($"Normes gradients W1={grad.W1.FrobeniusNorm():F6} b1={grad.b1.L2Norm():F6} W2={grad.W2.FrobeniusNorm():F6} b2={grad.b2.L2Norm():F6}");

Normes gradients W1=0.331920 b1=0.292304 W2=1.215565 b2=0.144167


***

## 4. Le contrôle du calcul : gradient numérique par différence finie

Écrire le backward à la main, c'est facile de se tromper d'un signe ou d'une transposée. La
vérification canonique : approcher chaque dérivée partielle par **différence centrale**
$\frac{L(\theta + \varepsilon) - L(\theta - \varepsilon)}{2\varepsilon}$ et comparer au
gradient analytique, **paramètre par paramètre**, sur les quatre tenseurs. Sur ces 20 exemples, on compare l’écart au seuil pratique $10^{-6}$ avec $\varepsilon=10^{-5}$. Le contrôle est local et peut être perturbé au voisinage d’un angle ReLU.

In [4]:
double GradientCheck(Params p,Matrix<double> x,long[] labels){var g=Backward(p,x,labels);double worst=0,eps=1e-5;void Check(Func<double> get,Action<double> set,double analytic){double old=get();set(old+eps);double plus=Loss(p,x,labels);set(old-eps);double minus=Loss(p,x,labels);set(old);worst=Math.Max(worst,Math.Abs(analytic-(plus-minus)/(2*eps)));}for(int i=0;i<2;i++)for(int j=0;j<32;j++){int r=i,c=j;Check(()=>p.W1[r,c],v=>p.W1[r,c]=v,g.W1[r,c]);}for(int i=0;i<32;i++){int j=i;Check(()=>p.b1[j],v=>p.b1[j]=v,g.b1[j]);}for(int i=0;i<32;i++)for(int j=0;j<2;j++){int r=i,c=j;Check(()=>p.W2[r,c],v=>p.W2[r,c]=v,g.W2[r,c]);}for(int i=0;i<2;i++){int j=i;Check(()=>p.b2[j],v=>p.b2[j]=v,g.b2[j]);}return worst;}
var chkX=Xtr.SubMatrix(0,20,0,2);var chkY=ytr.Take(20).ToArray();double err=GradientCheck(initial.Copy(),chkX,chkY);Console.WriteLine($"162 différences centrales, epsilon 1e-5 : écart max {err:E3} ; seuil 1e-6");if(err>=1e-6)throw new Exception("Gradient manuel");

162 différences centrales, epsilon 1e-5 : écart max 3.138E-011 ; seuil 1e-6


**Lecture.** Les 162 paramètres sont comparés à des différences centrales sur 20 exemples. Ce contrôle local, éloigné si possible des points anguleux de ReLU, ne prouve pas une formule pour toutes les entrées.


***

## 5. Boucle d'entraînement : le réseau résout ce que la droite ne pouvait pas

Il ne reste qu'à boucler : descente de gradient **pure** (le régime de 2.2 — momentum et Adam
sont le sujet de [3.2](./3.2-Optimisateurs.ipynb), pas d'ici), 3000 itérations full-batch, un learning rate
modeste. À chaque pas : forward, backward, mise à jour $\theta \leftarrow \theta -
\eta \nabla L$.

In [5]:
void Step(Params p,Params g,double lr){p.W1-=lr*g.W1;p.b1-=lr*g.b1;p.W2-=lr*g.W2;p.b2-=lr*g.b2;}
List<(int,double)> Train(Params p,Matrix<double> x,long[] labels,double lr=.1,int iters=3000,int logEvery=500){var h=new List<(int,double)>();for(int it=0;it<=iters;it++){if(it%logEvery==0)h.Add((it,Loss(p,x,labels)));if(it==iters)break;Step(p,Backward(p,x,labels),lr);}return h;}
double Accuracy(Params p,Matrix<double> x,long[] labels){var prob=Forward(p,x).P;int n=0;for(int i=0;i<labels.Length;i++)if((prob[i,1]>prob[i,0]?1:0)==labels[i])n++;return (double)n/labels.Length;}
var zeroStep=initial.Copy();Train(zeroStep,Xtr,ytr,iters:0);if((zeroStep.W1-initial.W1).FrobeniusNorm()!=0 || (zeroStep.W2-initial.W2).FrobeniusNorm()!=0 || (zeroStep.b1-initial.b1).L2Norm()!=0 || (zeroStep.b2-initial.b2).L2Norm()!=0)throw new Exception("Budget zéro");var trained=initial.Copy();var hist=Train(trained,Xtr,ytr);Console.WriteLine($"3000 mises à jour : loss {hist[0].Item2:F6} → {hist[^1].Item2:F6} ; train {Accuracy(trained,Xtr,ytr):F3}, test {Accuracy(trained,Xte,yte):F3}");
double DecisionNet(double x,double y){var a=DenseMatrix.OfArray(new double[,]{{x,y}});var f=Forward(trained,a);return f.Z2[0,1]-f.Z2[0,0];}
string PanelPair(string left,string right)=>"<svg xmlns='http://www.w3.org/2000/svg' width='1280' height='450' viewBox='0 0 1280 450'><rect width='1280' height='450' fill='white'/>"+left.Replace("<svg ","<svg x='0' y='0' ")+right.Replace("<svg ","<svg x='640' y='0' ")+"</svg>";
display(new PlotSvg(PanelPair(Curve("Entraînement manuel",hist),Decision("Frontière du MLP — données de test",xteRows,yte,DecisionNet))));

3000 mises à jour : loss 1.023154 → 0.108114 ; train 0.964, test 0.973


Entraînement manuel 0 0 750 0.269 1500 0.537 2250 0.806 3000 1.074 itération cross-entropy — manuel Frontière du MLP — données de test -1.865 -1.531 -0.668 -0.637 0.53 0.258 1.727 1.153 2.924 2.047 x1 x2 ● classe 0 ● classe 1 ● frontière de décision

**Lecture.** Sur ce run C#, la perte baisse de **1,0232 à 0,1081** en 3000 mises à jour ; la précision vaut **0,964** en entraînement et **0,973** en test, contre **0,873** pour la baseline logistique. La frontière apprise est courbe. La graine C# diffère de NumPy ; le témoin autograd part des mêmes poids que le modèle manuel C#.


## Exercice 1 : tanh au lieu de ReLU — la dérivée locale change

Implémentez Xavier, forward et backward tanh (dérivée `1 - tanh²`), vérifiez numériquement les quatre gradients et entraînez le modèle. Les stubs `InitXavier`, `ForwardTanh`, `BackwardTanh` reprennent les structures du modèle principal.


`TrainTanh` retourne les paramètres finaux et les checkpoints ; les fixtures contrôlent la non-linéarité, les gradients et la cohérence de la dernière perte.


In [6]:
// Exercice 1 : Xavier, tanh et descente ; les quatre stubs restent à compléter.
Params InitXavier(int seed)=>null;
ForwardResult ForwardTanh(Params p,Matrix<double> x)=>null;
Params BackwardTanh(Params p,Matrix<double> x,long[] labels)=>null;
(Params,List<(int,double)>)? TrainTanh(Params p,Matrix<double> x,long[] labels,int iters)=>null;
double TanhLoss(Params p,Matrix<double> x,long[] labels){var z=ForwardTanh(p,x).Z2;double total=0;for(int i=0;i<x.RowCount;i++){double m=Math.Max(z[i,0],z[i,1]);total+=Math.Log(Math.Exp(z[i,0]-m)+Math.Exp(z[i,1]-m))-(z[i,(int)labels[i]]-m);}return total/x.RowCount;}
double EcartType(Matrix<double> m){var vals=m.Enumerate().ToArray();double mean=vals.Average();return Math.Sqrt(vals.Sum(v=>(v-mean)*(v-mean))/vals.Length);}
var tanh=InitXavier(42);if(tanh==null)Console.WriteLine("Exercice 1 à compléter");else {try {var f=ForwardTanh(tanh,chkX);var g=BackwardTanh(tanh,chkX,chkY);bool valid=f!=null&&g!=null&&f.P.RowCount==20&&f.P.ColumnCount==2&&g.W1.RowCount==2&&g.W1.ColumnCount==32&&g.W2.RowCount==32&&g.W2.ColumnCount==2 && EcartType(tanh.W1)>.2 && EcartType(tanh.W1)<2 && EcartType(tanh.W2)>.03 && EcartType(tanh.W2)<.6 && tanh.b1.L2Norm()==0 && tanh.b2.L2Norm()==0;for(int i=0;i<20;i++)for(int j=0;j<32;j++)valid &= Math.Abs(f.A1[i,j]-Math.Tanh((chkX*tanh.W1)[i,j]+tanh.b1[j]))<1e-9;
 var expectedZ=f.A1*tanh.W2;for(int i=0;i<20;i++){for(int j=0;j<2;j++){expectedZ[i,j]+=tanh.b2[j];valid &= Math.Abs(f.Z2[i,j]-expectedZ[i,j])<1e-9;}double m=Math.Max(expectedZ[i,0],expectedZ[i,1]),e0=Math.Exp(expectedZ[i,0]-m),e1=Math.Exp(expectedZ[i,1]-m);valid &= Math.Abs(f.P[i,0]-e0/(e0+e1))<1e-9&&Math.Abs(f.P[i,1]-e1/(e0+e1))<1e-9;}
 void Check(Func<double> get,Action<double> set,double analytic){double old=get();set(old+1e-5);double a=TanhLoss(tanh,chkX,chkY);set(old-1e-5);double b=TanhLoss(tanh,chkX,chkY);set(old);valid &= double.IsFinite(analytic)&&Math.Abs(analytic-(a-b)/2e-5)<1e-6;}
 for(int j=0;j<3;j++){int k=j;Check(()=>tanh.W1[0,k],v=>tanh.W1[0,k]=v,g.W1[0,k]);Check(()=>tanh.b1[k],v=>tanh.b1[k]=v,g.b1[k]);Check(()=>tanh.W2[k,0],v=>tanh.W2[k,0]=v,g.W2[k,0]);}Check(()=>tanh.b2[0],v=>tanh.b2[0]=v,g.b2[0]);var shortTanh=TrainTanh(tanh.Copy(),chkX,chkY,2);valid &= shortTanh.HasValue;if(shortTanh.HasValue){var reference=tanh.Copy();for(int it=0;it<2;it++)Step(reference,BackwardTanh(reference,chkX,chkY),.1);var actual=shortTanh.Value.Item1;valid &= (actual.W1-reference.W1).FrobeniusNorm()<1e-8&&(actual.W2-reference.W2).FrobeniusNorm()<1e-8&&(actual.b1-reference.b1).L2Norm()<1e-8&&(actual.b2-reference.b2).L2Norm()<1e-8;}var trainedTanh=TrainTanh(tanh.Copy(),Xtr,ytr,3000);valid &= trainedTanh.HasValue;if(trainedTanh.HasValue){var (p,h)=trainedTanh.Value;valid &= h.Count>=2&&h[0].Item1==0&&h[^1].Item1==3000&&Math.Abs(h[0].Item2-TanhLoss(tanh,Xtr,ytr))<1e-8&&Math.Abs(h[^1].Item2-TanhLoss(p,Xtr,ytr))<1e-8 && h[^1].Item2<h[0].Item2;}Console.WriteLine(valid?"Fixtures tanh validées":"Réponse incorrecte");}catch(Exception){Console.WriteLine("Réponse incorrecte");}}

Exercice 1 à compléter


***

## 6. La comparaison avec autograd : PyTorch, mêmes poids, même pas

Le même réseau en PyTorch — mêmes dimensions, **mêmes poids initiaux** (copiés depuis C#,
en float64 pour une comparaison numérique) — et trois mesures : la loss au premier forward, les
poids après un pas de descente, puis la trajectoire complète des 3000 itérations. Si nos maths
sont celles de PyTorch, on mesure les écarts.

In [7]:
// Autograd TorchSharp : chaque bras redémarre depuis les mêmes poids.
Tensor TT(Matrix<double> x,bool requires=false)=>torch.tensor(x.ToArray(),dtype:ScalarType.Float64,requires_grad:requires);
Tensor TV(Vector<double> x,bool requires=false)=>torch.tensor(x.ToArray(),dtype:ScalarType.Float64,requires_grad:requires);
Tensor[] ToTorch(Params p)=>new[]{TT(p.W1,true),TV(p.b1,true),TT(p.W2,true),TV(p.b2,true)};
Tensor TorchLoss(Tensor[] p,Tensor xx,Tensor yy){var z1=xx.matmul(p[0])+p[1];var hidden=torch.nn.functional.relu(z1);var z2=hidden.matmul(p[2])+p[3];return torch.nn.functional.cross_entropy(z2,yy);}
var xt=TT(Xtr);var yt=torch.tensor(ytr,dtype:ScalarType.Int64);
using(var extremeLogits=torch.tensor(new double[,]{{1000,-1000},{1000,-1000}}))using(var extremeTargets=torch.tensor(new long[]{0,1}))using(var extremeLoss=torch.nn.functional.cross_entropy(extremeLogits,extremeTargets)){double lossExtreme=extremeLoss.item<double>();if(Math.Abs(lossExtreme-1000)>1e-10)throw new Exception("Cross-entropy extrême TorchSharp");Console.WriteLine($"TorchSharp cross-entropy des logits extrêmes : {lossExtreme:F1} (moyenne 0 et 2000)");}
using(var shifted=torch.tensor(new double[,]{{1e20,1e20}}))using(var label=torch.tensor(new long[]{0}))using(var shiftedLoss=torch.nn.functional.cross_entropy(shifted,label)){double oracle=shiftedLoss.item<double>();if(Math.Abs(oracle-Math.Log(2))>1e-15||Math.Abs(oracle-Loss(extreme,one,new long[]{0}))>1e-15)throw new Exception("Décalage commun TorchSharp");Console.WriteLine($"TorchSharp, logits décalés : {oracle:F12}");}
var tp=ToTorch(initial);var tLoss=TorchLoss(tp,xt,yt);double firstLoss=tLoss.item<double>();Console.WriteLine($"Loss initiale : manuel {Loss(initial,Xtr,ytr):F12}, TorchSharp {firstLoss:F12}, écart {Math.Abs(firstLoss-Loss(initial,Xtr,ytr)):E3}");tLoss.backward();var manualStep=initial.Copy();Step(manualStep,Backward(manualStep,Xtr,ytr),.1);using(torch.no_grad())foreach(var t in tp)t.sub_(.1*t.grad!);double[] flatten(Matrix<double> m)=>m.ToRowMajorArray();double[][] expected={flatten(manualStep.W1),manualStep.b1.ToArray(),flatten(manualStep.W2),manualStep.b2.ToArray()};double maxParam=0;for(int k=0;k<4;k++){var vals=tp[k].data<double>().ToArray();for(int j=0;j<vals.Length;j++)maxParam=Math.Max(maxParam,Math.Abs(vals[j]-expected[k][j]));}Console.WriteLine($"Après un pas, écart maximal sur les 162 paramètres : {maxParam:E3}");if(maxParam>1e-9)throw new Exception("Pas autograd divergent");
List<(int,double)> TrainTorch(Params p){var tensors=ToTorch(p);var history=new List<(int,double)>();for(int it=0;it<=3000;it++){using var scope=torch.NewDisposeScope();var loss=TorchLoss(tensors,xt,yt);if(it%500==0)history.Add((it,loss.item<double>()));if(it==3000)break;loss.backward();using(torch.no_grad())foreach(var t in tensors)t.sub_(.1*t.grad!);foreach(var t in tensors)t.grad=null;}foreach(var t in tensors)t.Dispose();return history;}
var torchHist=TrainTorch(initial);double trajectory=hist.Zip(torchHist,(a,b)=>Math.Abs(a.Item2-b.Item2)).Max();Console.WriteLine($"Trajectoire 3000 mises à jour : écart maximal {trajectory:E3}");if(trajectory>1e-8)throw new Exception("Trajectoire divergente");display(new PlotSvg(Curve("Trajectoires : manuel et autograd",hist,torchHist)));

TorchSharp cross-entropy des logits extrêmes : 1000.0 (moyenne 0 et 2000)


TorchSharp, logits décalés : 0.693147180560


Loss initiale : manuel 1.023154386769, TorchSharp 1.023154386769, écart 2.220E-016


Après un pas, écart maximal sur les 162 paramètres : 5.551E-017


Trajectoire 3000 mises à jour : écart maximal 2.220E-016


Trajectoires : manuel et autograd 0 0 750 0.269 1500 0.537 2250 0.806 3000 1.074 itération cross-entropy — manuel — TorchSharp autograd


warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'TorchSharp' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply r

**Lecture.** Dans ce run float64, l’écart de loss initiale est **2,22 × 10⁻¹⁶**, l’écart maximal sur les 162 paramètres après un pas **5,55 × 10⁻¹⁷**, et l’écart maximal des sept checkpoints **2,22 × 10⁻¹⁶**. Les trajectoires coïncident à la précision mesurée pour cette expérience, sans garantir une identité bit à bit sur d’autres données ou matériels.


## Exercice 2 : cercles concentriques et deuxième couche cachée

Générez 600 exemples avec `make_circles(factor=0.4, noise=0.15, random_state=42)` via sklearn. Construisez et entraînez un réseau 2→32→32→2 ; vérifiez les gradients sur un petit échantillon avant l’entraînement.


`CircleSolution` expose paramètres finaux et initiaux, fonctions forward/backward et historique ; les fixtures comparent les gradients aux différences finies et les pertes initiale et finale aux paramètres rendus ; la perte finale doit être inférieure à ln(2).


In [8]:
// Exercice 2 : le modèle et ses fonctions doivent provenir de l'exercice étudiant.
class CircleParams { public Matrix<double> W1,W2,W3; public Vector<double> b1,b2,b3; }
class CircleOutput { public Matrix<double> P,Z3; }
class CircleSolution { public CircleParams P,Initial; public Func<CircleParams,Matrix<double>,CircleOutput> Forward; public Func<CircleParams,Matrix<double>,long[],CircleParams> Backward; public List<double> History; }
CircleSolution EntrainerCercles()=>null; // TODO : 600 points, 2→32→32→2, gradient puis entraînement
var circles=EntrainerCercles();if(circles==null)Console.WriteLine("Exercice 2 à compléter");else {try {double[][] rows=null;long[] labels=null;using(Py.GIL()){dynamic ds=Py.Import("sklearn.datasets"),ms=Py.Import("sklearn.model_selection");using PyObject pair=ds.make_circles(n_samples:600,factor:.4,noise:.15,random_state:42);using var a=pair.GetItem(0);using var b=pair.GetItem(1);using PyObject split=ms.train_test_split(a,b,test_size:.25,random_state:42,stratify:b);using var x=split.GetItem(0);using var y=split.GetItem(2);using var xr=x.InvokeMethod("tolist");using var yr=y.InvokeMethod("tolist");rows=xr.As<double[][]>();labels=yr.As<long[]>();}var xall=FromRows(rows);var xs=xall.SubMatrix(0,20,0,2);var ys=labels.Take(20).ToArray();var p=circles.P;var f=circles.Forward(p,xs);var g=circles.Backward(p,xs,ys);bool valid=p.W1.RowCount==2&&p.W1.ColumnCount==32&&p.W2.RowCount==32&&p.W2.ColumnCount==32&&p.W3.RowCount==32&&p.W3.ColumnCount==2&&f.P.RowCount==20&&f.P.ColumnCount==2;
 for(int i=0;i<20;i++){var z1=new double[32];var z2=new double[32];for(int j=0;j<32;j++){z1[j]=Math.Max(0,xs[i,0]*p.W1[0,j]+xs[i,1]*p.W1[1,j]+p.b1[j]);}for(int j=0;j<32;j++){double z=p.b2[j];for(int k=0;k<32;k++)z+=z1[k]*p.W2[k,j];z2[j]=Math.Max(0,z);}for(int j=0;j<2;j++){double z=p.b3[j];for(int k=0;k<32;k++)z+=z2[k]*p.W3[k,j];valid &= Math.Abs(f.Z3[i,j]-z)<1e-9;}double m=Math.Max(f.Z3[i,0],f.Z3[i,1]),e0=Math.Exp(f.Z3[i,0]-m),e1=Math.Exp(f.Z3[i,1]-m);valid &= Math.Abs(f.P[i,0]-e0/(e0+e1))<1e-9&&Math.Abs(f.P[i,1]-e1/(e0+e1))<1e-9;}
 double CE(CircleParams q,Matrix<double> x,long[] y){var z=circles.Forward(q,x).Z3;double sum=0;for(int i=0;i<x.RowCount;i++){double m=Math.Max(z[i,0],z[i,1]);sum+=Math.Log(Math.Exp(z[i,0]-m)+Math.Exp(z[i,1]-m))-(z[i,(int)y[i]]-m);}return sum/x.RowCount;}
 void Check(Func<double> get,Action<double> set,double analytic){double old=get();set(old+1e-5);double a=CE(p,xs,ys);set(old-1e-5);double b=CE(p,xs,ys);set(old);valid &= double.IsFinite(analytic)&&Math.Abs(analytic-(a-b)/2e-5)<1e-6;}
 Check(()=>p.W1[0,0],v=>p.W1[0,0]=v,g.W1[0,0]);Check(()=>p.b1[0],v=>p.b1[0]=v,g.b1[0]);Check(()=>p.W2[0,0],v=>p.W2[0,0]=v,g.W2[0,0]);Check(()=>p.b2[0],v=>p.b2[0]=v,g.b2[0]);Check(()=>p.W3[0,0],v=>p.W3[0,0]=v,g.W3[0,0]);Check(()=>p.b3[0],v=>p.b3[0]=v,g.b3[0]);valid &= circles.History.Count>=2 && circles.History.All(double.IsFinite) && Math.Abs(circles.History[^1]-CE(p,xall,labels))<1e-7 && Math.Abs(circles.History[0]-CE(circles.Initial,xall,labels))<1e-7 && circles.History[^1]<Math.Log(2)-1e-8 && circles.History[^1]<circles.History[0];Console.WriteLine(valid?"Fixtures cercles validées":"Réponse incorrecte");}catch(Exception){Console.WriteLine("Réponse incorrecte");}}

Exercice 2 à compléter



(4,46): warning CS0649: Field 'CircleSolution.Initial' is never assigned to, and will always have its default value null

(3,46): warning CS0649: Field 'CircleOutput.Z3' is never assigned to, and will always have its default value null

(4,109): warning CS0649: Field 'CircleSolution.Forward' is never assigned to, and will always have its default value null

(2,82): warning CS0649: Field 'CircleParams.b3' is never assigned to, and will always have its default value null

(4,209): warning CS0649: Field 'CircleSolution.History' is never assigned to, and will always have its default value null

(3,44): warning CS0649: Field 'CircleOutput.P' is never assigned to, and will always have its default value null

(4,179): warning CS0649: Field 'CircleSolution.Backward' is never assigned to, and will always have its default value null

(2,76): warning CS0649: Field 'CircleParams.b1' is never assigned to, and will always have its default value null

(2,44): warning CS0649: Field 'CircleParams.W1' 

***

## 7. Initialisation nulle : un réseau ReLU sur ce split équilibré

Dernière expérience, contre-intuitive : et si l'on initialisait tous les poids à **zéro** ?
La descente de gradient saura bien les corriger, non ? Non — et la backward pass du §3 explique
pourquoi : chaque neurone d'une couche reçoit alors **exactement le même gradient**, reste
identique à ses voisins, et la symétrie n'est jamais brisée. Vérifions-le en mesurant la norme
du gradient des poids.

In [9]:
var zero=initial.Copy();zero.W1.Clear();zero.W2.Clear();zero.b1.Clear();zero.b2.Clear();var gz=Backward(zero,Xtr,ytr);Console.WriteLine($"Initialisation nulle : ||dW1||={gz.W1.FrobeniusNorm():E3}, ||dW2||={gz.W2.FrobeniusNorm():E3}");var hz=Train(zero,Xtr,ytr);Console.WriteLine($"Après 3000 : loss={hz[^1].Item2:F6}, test={Accuracy(zero,Xte,yte):F3} ; He={hist[^1].Item2:F6}");

Initialisation nulle : ||dW1||=0.000E+000, ||dW2||=0.000E+000


Après 3000 : loss=0.693147, test=0.500 ; He=0.108114


**Lecture.** Sur ce split équilibré et cette architecture ReLU, les deux gradients de poids sont nuls et la loss reste proche de ln 2. Une initialisation déterministe asymétrique peut aussi briser la symétrie ; un split déséquilibré peut donner un gradient du biais de sortie non nul.


## Exercice 3 : les trois régimes du learning rate

Rejouez 2000 mises à jour depuis les mêmes poids He pour 0,01, 0,1 et 10. Tracez les courbes et décrivez le comportement observé, sans imposer une divergence.


`ExperiencesLR` retourne les trois modèles et checkpoints : les fixtures rejouent les 2000 pas pour confronter chaque état.


In [10]:
// Exercice 3 : retourner les paramètres et les checkpoints à 0, 500, ..., 2000.
Dictionary<double,(Params,List<(int,double)>)> ExperiencesLR()=>null;
var lrResult=ExperiencesLR();if(lrResult==null)Console.WriteLine("Exercice 3 à compléter");else {try {bool valid=lrResult.Count==3&&new[]{.01,.1,10.0}.All(lrResult.ContainsKey);foreach(double lr in new[]{.01,.1,10.0}){var (p,h)=lrResult[lr];var reference=initial.Copy();var expected=Train(reference,Xtr,ytr,lr,2000,500);valid &= h.Count==expected.Count;for(int i=0;i<Math.Min(h.Count,expected.Count);i++)valid &= h[i].Item1==expected[i].Item1&&double.IsFinite(h[i].Item2)&&Math.Abs(h[i].Item2-expected[i].Item2)<1e-7;valid &= (p.W1-reference.W1).FrobeniusNorm()<1e-7&&(p.W2-reference.W2).FrobeniusNorm()<1e-7&&(p.b1-reference.b1).L2Norm()<1e-7&&(p.b2-reference.b2).L2Norm()<1e-7;}Console.WriteLine(valid?"Fixtures taux validées":"Réponse incorrecte");}catch(Exception){Console.WriteLine("Réponse incorrecte");}}

Exercice 3 à compléter


***


## 8. Capstone : la backward comme composition de Jacobiennes

Pour le réseau sigmoid 2→4→1, calculez les quatre Jacobiennes et le produit vecteur-Jacobienne `dL/dx` pour `x=[0.8,-0.6]`, `y=1`. Fixez la BCE sur le logit `z` : $L=\log(1+e^z)-yz$. Vérifiez les deux composantes par différences finies avec ε=1e-5 et seuil 1e-7 ; testez aussi une deuxième entrée et classe.


In [11]:
// Exercice 4 : BCE sigmoid pour un réseau 2→4→1 et VJP dL/dx.
(double, double[])? CapstoneVJP(double[] x,double y,double[,] w1,double[] b1,double[] w2,double b2)=>null;
// Tirages gaussiens seed 0 ; Random C# et NumPy n’ont pas le même générateur.
var capRng=new Random(0);double CapNormal(){double u=1-capRng.NextDouble(),v=1-capRng.NextDouble();return Math.Sqrt(-2*Math.Log(u))*Math.Cos(2*Math.PI*v);}
var Wex=new double[2,4];for(int i=0;i<2;i++)for(int j=0;j<4;j++)Wex[i,j]=CapNormal();var bex=Enumerable.Range(0,4).Select(_=>CapNormal()).ToArray();var wex=Enumerable.Range(0,4).Select(_=>CapNormal()).ToArray();double b2ex=CapNormal();
double Logit(double[] x){double z=b2ex;for(int j=0;j<4;j++){double a=bex[j];for(int i=0;i<2;i++)a+=x[i]*Wex[i,j];z+=wex[j]/(1+Math.Exp(-a));}return z;}
double BCE(double[] x,double y){double z=Logit(x);return Math.Max(z,0)+Math.Log(1+Math.Exp(-Math.Abs(z)))-y*z;}
var cap=CapstoneVJP(new[]{.8,-.6},1,Wex,bex,wex,b2ex);if(cap==null)Console.WriteLine("Exercice 4 à compléter");else {try{bool valid=true;foreach(var (x,y) in new[]{(new[]{.8,-.6},1.0),(new[]{-.4,.2},0.0)}){var result=CapstoneVJP(x,y,Wex,bex,wex,b2ex);if(!result.HasValue){valid=false;continue;}var (prob,gradient)=result.Value;double expected=1/(1+Math.Exp(-Logit(x)));valid &= double.IsFinite(prob)&&Math.Abs(prob-expected)<1e-9&&gradient!=null&&gradient.Length==2;if(gradient!=null&&gradient.Length==2)for(int j=0;j<2;j++){var xp=(double[])x.Clone();var xm=(double[])x.Clone();xp[j]+=1e-5;xm[j]-=1e-5;double numeric=(BCE(xp,y)-BCE(xm,y))/2e-5;valid &= double.IsFinite(gradient[j])&&Math.Abs(gradient[j]-numeric)<1e-7;}}Console.WriteLine(valid?"Fixtures VJP validées":"Réponse incorrecte");}catch(Exception){Console.WriteLine("Réponse incorrecte");}}

Exercice 4 à compléter


## Contre-sens usuels sur la rétropropagation

- Une différence finie vérifie localement une implémentation ; ReLU n’est pas différentiable en zéro.
- La comparaison autograd porte sur les mêmes paramètres et les mêmes données, avec écarts numériques mesurés.
- Sur ce réseau ReLU et ce split équilibré, l’initialisation nulle fige les poids ; la symétrie peut aussi être brisée de façon déterministe.
- Une frontière non linéaire n’est pas réservée aux réseaux.


***


## Conclusion et transition

Le calcul manuel, les différences finies et l’autograd sont confrontés sur les mêmes données. Les sorties mesurent la perte, les gradients, les paramètres et la trajectoire ; l’exemple d’initialisation nulle illustre le rôle de la symétrie. Le capstone relie la BCE aux produits vecteur-Jacobienne. Poursuivez avec [3.2 — Optimisateurs](3.2-Optimisateurs.ipynb).


***


## References

1. Rumelhart, D., Hinton, G. & Williams, R. (1986). *Learning representations by back-propagating errors*. Nature 323, 533–536. — L'article fondateur : la rétropropagation appliquée aux réseaux multicouches.
2. Glorot, X. & Bengio, Y. (2010). *Understanding the difficulty of training deep feedforward neural networks*. AISTATS 2010. — Pourquoi l'initialisation compte : variance $1/\mathrm{fan\_in}$ et rupture de symétrie.
3. Nielsen, M. *Neural Networks and Deep Learning*, chapitre 2 — une dérivation pas à pas de la rétropropagation, avec le point de vue Jacobien.
4. Baydin, A., Pearlmutter, B., Radul, A. & Siskind, J. (2018). *Automatic Differentiation in Machine Learning: a Survey*. JMLR 18. — Forward vs reverse mode : ce qu'autograd fait en silence.
5. Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, ch. 6.5 « Back-Propagation and Other Differentiation Algorithms » (pp. 204–211). MIT Press. — Le traitement canonique moderne de la rétropropagation, après l'article fondateur de Rumelhart : dérivation tensorielle, notation par graphe de calcul, gestion des Jacobiens par produit vectoriel, et l'algorithme utilisé dans tous les frameworks (PyTorch `loss.backward()`, TensorFlow `GradientTape`). C'est la référence que cherche un étudiant qui veut approfondir après ce carnet.


## Vérifiez votre compréhension

1. Pourquoi calculer la cross-entropy à partir des logits plutôt que de `log(softmax)` ?
2. Que vérifie la différence centrale et que se passe-t-il près d’un angle de ReLU ?
3. Que compare le témoin autograd après un pas et sur les 3000 mises à jour ?
4. Pourquoi l’initialisation nulle bloque-t-elle ici les poids cachés, et pourquoi ce résultat n’est-il pas universel ?

<details><summary>Réponses</summary>

1. Le log-sum-exp évite `log(0)` et préserve l’objectif, même pour des logits extrêmes.
2. Elle contrôle localement les dérivées numériques ; au point anguleux, la dérivée de ReLU n’est pas unique.
3. Les pertes initiales, les quatre tenseurs après un pas et les checkpoints de loss depuis des poids identiques.
4. ReLU à zéro et les poids nuls bloquent ici les gradients cachés ; les biais de sortie peuvent avoir un gradient sur un split déséquilibré et une initialisation asymétrique peut être déterministe.

</details>


## Pour aller plus loin

- [PyTorch : cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) et [autograd](https://docs.pytorch.org/docs/stable/autograd.html).
- [scikit-learn : make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html).
- [Références du carnet](#references) : Rumelhart, Glorot, Baydin et Goodfellow pour les dérivations et l’optimisation.
